# Preparação dos Microdados do ENEM 2023

**Notebook auxiliar — documentação e reprodutibilidade**

Este notebook registra, de forma executável, como obter os [Microdados do ENEM 2023](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem), selecionar as 16 variáveis utilizadas no projeto **ENEM 2023 — Viés de Seleção Socioeconômico** e dividir os registros em 16 arquivos CSV.

> **Não é necessário executar este notebook para reproduzir a análise principal.** O arquivo original do INEP tem aproximadamente **1,74 GB** após a extração, e seu download e processamento podem consumir tempo e recursos consideráveis. Os **16 CSVs já preparados** estão disponíveis na pasta [`dados/`](https://github.com/calvariofilipe/enem-2023-vies-de-selecao/tree/main/dados) do repositório e são lidos diretamente pelo notebook principal. Este notebook auxiliar existe **apenas para documentar a origem e permitir a reprodução da etapa inicial de preparação**, caso isso seja desejado.

**Fonte:** INEP — Microdados do ENEM, edição 2023  
**Repositório:** https://github.com/calvariofilipe/enem-2023-vies-de-selecao

A seleção reduz o **número de colunas**, mas **não elimina inscrições**: participantes presentes, ausentes e eliminados continuam na base. A análise exploratória e as decisões de pré-processamento são realizadas no notebook principal.

## 1. Configuração e variáveis selecionadas

São mantidas 16 colunas: identificador de inscrição; faixa etária, tipo de escola e UF da prova; escolaridade dos responsáveis e renda familiar; indicadores de presença; e notas das quatro provas objetivas e da redação.

A leitura será feita **registro por registro** com a biblioteca nativa `csv`, usando o separador `;` e a codificação `latin-1` do arquivo de origem. Isso evita colocar o CSV original inteiro na memória.

In [ ]:
from pathlib import Path
import csv
import io
import zipfile
from collections import Counter

PASTA_BASE = Path('/content') if Path('/content').exists() else Path.cwd()
ZIP = PASTA_BASE / 'microdados_enem_2023.zip'
PASTA_SAIDA = PASTA_BASE / 'dados'
LINHAS_POR_ARQUIVO = 250_000
TOTAL_ESPERADO = 3_933_955

URL = 'https://download.inep.gov.br/microdados/microdados_enem_2023.zip'

COLUNAS = [
    'NU_INSCRICAO', 'TP_FAIXA_ETARIA', 'TP_ESCOLA', 'SG_UF_PROVA',
    'Q001', 'Q002', 'Q006', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH',
    'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH',
    'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO'
]

print(f'Pasta de trabalho: {PASTA_BASE}')
print(f'Colunas selecionadas: {len(COLUNAS)}')
print(f'Máximo de registros por arquivo: {LINHAS_POR_ARQUIVO:,}')

## 2. Obtenção dos dados oficiais

O código abaixo baixa o ZIP diretamente do INEP, com **barra de progresso** e possibilidade de **retomar uma transferência parcial** quando o servidor oferece suporte. Se o ZIP já estiver presente e puder ser aberto, não será baixado novamente.

O servidor do INEP pode apresentar lentidão ou problemas de certificado/conexão. Nesse caso, é possível baixar o ZIP pelo [site oficial](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) e enviá-lo ao ambiente com o nome `microdados_enem_2023.zip`. A verificação de certificado HTTPS permanece habilitada por segurança.

**Observação:** esta é a etapa mais demorada e **dispensável** para quem deseja somente executar o MVP principal.

In [ ]:
# Caso o ZIP já tenha sido enviado manualmente, esta célula reaproveita o arquivo.
import requests
from tqdm.auto import tqdm


def zip_valido(caminho):
    return caminho.exists() and zipfile.is_zipfile(caminho)


def baixar_zip(url, destino):
    if zip_valido(destino):
        print(f'ZIP já disponível: {destino} ({destino.stat().st_size / 2**20:.1f} MiB)')
        return

    tamanho_existente = destino.stat().st_size if destino.exists() else 0
    headers = {'Range': f'bytes={tamanho_existente}-'} if tamanho_existente else {}
    print('Solicitando download ao INEP (pode haver demora na conexão)...', flush=True)

    try:
        with requests.get(url, headers=headers, stream=True, timeout=(30, 180)) as resposta:
            resposta.raise_for_status()
            # Só anexar ao arquivo se o servidor confirmar a retomada (HTTP 206).
            retomada = tamanho_existente > 0 and resposta.status_code == 206
            modo = 'ab' if retomada else 'wb'
            if tamanho_existente and not retomada:
                print('Servidor não confirmou retomada; iniciando download do começo.')
            total = int(resposta.headers.get('Content-Length', 0))
            with open(destino, modo) as arquivo, tqdm(
                total=(tamanho_existente + total) if total else None,
                initial=tamanho_existente if retomada else 0,
                unit='B', unit_scale=True, unit_divisor=1024,
                desc='Microdados ENEM 2023'
            ) as barra:
                for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                    if bloco:
                        arquivo.write(bloco)
                        barra.update(len(bloco))
    except requests.RequestException as erro:
        raise RuntimeError(
            'Falha no download do INEP. É possível enviar o ZIP manualmente '
            f'para {destino} e executar a célula novamente. Detalhes: {erro}'
        ) from erro

    if not zip_valido(destino):
        raise RuntimeError('O arquivo baixado não é um ZIP válido. Verifique a transferência.')
    print(f'ZIP pronto: {destino.stat().st_size / 2**20:.1f} MiB')


baixar_zip(URL, ZIP)

## 3. Extração das colunas e divisão em arquivos

O CSV é lido **diretamente de dentro do ZIP**: não é necessário extrair os 1,74 GB para o disco. O programa identifica as posições das 16 variáveis no cabeçalho, copia apenas esses campos de cada registro e cria um novo arquivo a cada **250 mil inscrições**.

Cada parte recebe um cabeçalho próprio e é gravada em **UTF-8**, mantendo o separador `;` e os valores das colunas selecionadas. Nenhum filtro por presença ou disponibilidade de nota é aplicado. A última parte terá menos registros que as anteriores.

Os arquivos resultantes são escritos em `dados/`, com nomes de `enem_2023_parte_001.csv` até `enem_2023_parte_016.csv`. Reexecutar esta célula **substitui os arquivos com esses nomes** na pasta de saída.

In [ ]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
total = 0
quantidade_arquivos = 0
presenca_mt = Counter()
notas_mt_disponiveis = 0
saida_aberta = None

with zipfile.ZipFile(ZIP) as zip_in:
    candidatos = [
        nome for nome in zip_in.namelist()
        if nome.upper().endswith('MICRODADOS_ENEM_2023.CSV')
    ]
    if len(candidatos) != 1:
        raise ValueError(f'Esperado um CSV original dentro do ZIP; encontrados: {candidatos}')

    with zip_in.open(candidatos[0]) as arquivo_binario:
        with io.TextIOWrapper(arquivo_binario, encoding='latin-1', newline='') as entrada:
            leitor = csv.reader(entrada, delimiter=';')
            cabecalho = next(leitor)
            faltantes = [coluna for coluna in COLUNAS if coluna not in cabecalho]
            if faltantes:
                raise ValueError(f'Colunas não encontradas no arquivo original: {faltantes}')
            posicoes = [cabecalho.index(coluna) for coluna in COLUNAS]
            indice_presenca = COLUNAS.index('TP_PRESENCA_MT')
            indice_nota = COLUNAS.index('NU_NOTA_MT')

            try:
                for linha in leitor:
                    if len(linha) != len(cabecalho):
                        raise ValueError(f'Registro {total + 1} com número inesperado de campos.')

                    if total % LINHAS_POR_ARQUIVO == 0:
                        if saida_aberta is not None:
                            saida_aberta.close()
                        quantidade_arquivos += 1
                        caminho = PASTA_SAIDA / f'enem_2023_parte_{quantidade_arquivos:03d}.csv'
                        saida_aberta = open(caminho, 'w', encoding='utf-8', newline='')
                        escritor = csv.writer(saida_aberta, delimiter=';')
                        escritor.writerow(COLUNAS)
                        print(f'Criando {caminho.name}', flush=True)

                    selecionados = [linha[i] for i in posicoes]
                    escritor.writerow(selecionados)
                    presenca_mt[selecionados[indice_presenca]] += 1
                    notas_mt_disponiveis += bool(selecionados[indice_nota].strip())
                    total += 1

                    if total % 250_000 == 0:
                        print(f'  {total:,} inscrições processadas', flush=True)
            finally:
                if saida_aberta is not None:
                    saida_aberta.close()

print(f'Extração encerrada: {total:,} registros em {quantidade_arquivos} arquivos.')

## 4. Verificação dos arquivos gerados

A checagem abaixo confirma a quantidade de partes e de registros, a presença das 16 colunas e a distribuição do número de linhas entre os arquivos. Também apresenta as frequências de presença em Matemática (`0` = ausente; `1` = presente; `2` = eliminado), permitindo confrontar os totais com os do projeto.

Esta verificação não substitui a análise exploratória de qualidade realizada no notebook principal; ela verifica a integridade estrutural **da etapa de extração**.

In [ ]:
arquivos = sorted(PASTA_SAIDA.glob('enem_2023_parte_*.csv'))
contagens = []

for arquivo in arquivos:
    with open(arquivo, 'r', encoding='utf-8', newline='') as entrada:
        leitor = csv.reader(entrada, delimiter=';')
        if next(leitor) != COLUNAS:
            raise AssertionError(f'Cabeçalho inesperado em {arquivo.name}')
        quantidade = sum(1 for _ in leitor)
        contagens.append(quantidade)

print('Arquivos:', len(arquivos))
print('Linhas por arquivo:', contagens)
print('Total conferido:', f'{sum(contagens):,}')
print('Tamanho total:', f'{sum(a.stat().st_size for a in arquivos) / 2**20:.2f} MiB')
print('Presença em Matemática:', dict(sorted(presenca_mt.items())))
print('Notas de Matemática disponíveis:', f'{notas_mt_disponiveis:,}')

assert len(arquivos) == 16, 'Quantidade de arquivos diferente de 16.'
assert total == TOTAL_ESPERADO == sum(contagens), 'Quantidade de registros divergente.'
assert all(n == LINHAS_POR_ARQUIVO for n in contagens[:-1]), 'Partes intermediárias incompletas.'
assert 0 < contagens[-1] <= LINHAS_POR_ARQUIVO, 'Última parte inválida.'
assert presenca_mt['0'] == 1_239_316
assert presenca_mt['1'] == 2_692_427
assert presenca_mt['2'] == 2_212
assert notas_mt_disponiveis == 2_692_427

print('Validações concluídas com sucesso.')

## 5. Utilização no MVP principal

Após a preparação, os CSVs podem ser publicados na pasta `dados/` do GitHub. O **notebook principal já utiliza os arquivos publicados**, carregando e concatenando as 16 partes — **não requer a execução deste notebook auxiliar**.

A finalidade deste material é tornar transparente e reproduzível a passagem da base oficial do INEP para a versão reduzida empregada na investigação. Os critérios de participação, agrupamentos de renda e demais decisões analíticas são tratados separadamente no MVP principal.

**Referência:** Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP). *Microdados do ENEM 2023*. https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem